# Smart Energy Consumption Forecasting & Anomaly Detection

## Part A: Exploratory Data Analysis and Feature Engineering

This notebook performs exploratory data analysis (EDA), feature engineering, and statistical analysis on the Appliances Energy Prediction dataset. The goal is to understand patterns in energy consumption and prepare features for predictive modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

The dataset is loaded and initial inspection is performed to understand its structure.

In [ ]:
df = pd.read_csv("energydata_complete.csv")
df.head()

## Dataset Overview

This dataset contains energy consumption data along with environmental and temporal features recorded at 10-minute intervals.

In [ ]:
df.info()
df.describe()

In [ ]:
#inorder to extract hours,day,week 
df['date'] = pd.to_datetime(df['date'])


df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

### 1 Energy Consumption Over Time (Sampled Data)

To visualize the overall trend of energy consumption, we plot the `Appliances` energy usage against time. Since the dataset contains high-frequency readings (every 10 minutes), plotting all points can be computationally expensive and cluttered.

To improve performance and readability, we sample every 10th data point using `df.iloc[::10]`.

This plot helps in identifying general consumption patterns, trends, and potential anomalies over time.


In [ ]:
df_sample = df.iloc[::10]  # sampling to speed up

plt.figure(figsize=(12,5))
plt.plot(df_sample['date'], df_sample['Appliances'])
plt.title("Energy Consumption Over Time")
plt.xlabel("Time")
plt.ylabel("Energy (Wh)")
plt.grid(True)
plt.show()

### 2. Energy Consumption by Hour of Day

Energy consumption varies significantly across different hours of the day, with higher values typically observed during daytime and evening hours. This pattern reflects human activity cycles, such as cooking, working, and appliance usage. The variation within each hour also indicates inconsistent usage patterns. This suggests that the hour of day is a crucial feature for modeling energy consumption.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='hour', y='Appliances', data=df)
plt.title("Energy Consumption by Hour of Day")
plt.show()

### 3. Energy Consumption by Day of Week

The energy consumption shows variation across different days of the week, indicating possible weekly usage patterns. Some days exhibit slightly higher median consumption, which may be due to differences in occupancy or lifestyle behavior. However, the variation is not extremely large, suggesting relatively consistent usage. Including day-of-week as a feature may still help capture subtle behavioral differences.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='day_of_week', y='Appliances', data=df)
plt.title("Energy Consumption by Day of Week")
plt.xlabel("Day of Week (0=Monday)")
plt.show()

### 4. Rolling Mean of Energy Consumption

The rolling mean smooths short-term fluctuations in energy consumption and helps reveal the underlying trend. This makes it easier to identify consistent patterns that may not be visible in raw data. The presence of smooth variations indicates predictable short-term behavior. Rolling statistics are useful for capturing local trends and improving model stability.

In [ ]:
df['rolling_mean_10'] = df['Appliances'].rolling(window=10).mean()

df_sample = df.iloc[::10]

plt.figure(figsize=(12,5))
plt.plot(df_sample['date'], df_sample['rolling_mean_10'])
plt.title("Rolling Mean of Energy Consumption")
plt.show()

### 5. Distribution of Energy Consumption

The distribution of energy consumption appears to be right-skewed, with most values concentrated at lower levels and a few extreme high values. This indicates that high energy usage events are relatively rare but significant. Such skewness can impact model performance, especially for algorithms sensitive to distribution. Understanding this distribution is important for selecting appropriate modeling techniques.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Appliances'], bins=50, kde=True)
plt.title("Distribution of Energy Consumption")
plt.show()

### 6. Boxplot of Energy Consumption

The boxplot highlights the presence of several outliers in energy consumption, indicating occasional extreme usage values. These outliers may be caused by unusual events or peak activity periods. Such extreme values can influence model performance and may require careful handling. Identifying outliers is important for building robust predictive models.

In [ ]:
plt.figure(figsize=(6,5))
sns.boxplot(y=df['Appliances'])
plt.title("Boxplot of Energy Consumption")
plt.show()

### 7. Temperature vs Energy Consumption

The scatter plot shows a positive relationship between temperature and energy consumption, suggesting that energy usage increases as temperature rises. This may be due to increased cooling demand during warmer conditions. However, the relationship is not strictly linear, indicating the influence of additional factors. Temperature is likely a significant feature for predicting energy consumption.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='T2', y='Appliances', data=df, alpha=0.3)
plt.title("Temperature vs Energy Consumption")
plt.show()

### 8. Humidity vs Energy Consumption

The relationship between humidity and energy consumption appears to be weaker compared to temperature. While some variation exists, no strong trend is observed. This suggests that humidity alone may not be a primary driver of energy usage. However, it could still contribute in combination with other environmental factors.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='RH_2', y='Appliances', data=df, alpha=0.3)
plt.title("Humidity vs Energy Consumption")
plt.show()

### 9. Correlation Heatmap

The correlation heatmap provides an overview of linear relationships between features. Temperature-related features show moderate correlation with energy consumption, indicating their relevance. However, many variables exhibit low correlation, suggesting the presence of non-linear relationships. This highlights the need for additional analysis methods such as mutual information.

In [ ]:
plt.figure(figsize=(12,8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

### 10. Lag Plot of Energy Consumption

The lag plot shows a strong relationship between previous and current energy values, indicating temporal dependency in the data. This suggests that past energy consumption significantly influences future values. Such behavior is typical in time-series data and justifies the use of lag features. This insight supports the use of sequence-based models and time-aware feature engineering.

In [ ]:
df['lag_1'] = df['Appliances'].shift(1)

df_lag = df[['lag_1', 'Appliances']].dropna().iloc[::10]

plt.figure(figsize=(8,5))
sns.scatterplot(x='lag_1', y='Appliances', data=df_lag, alpha=0.3)
plt.title("Lag Plot (Previous vs Current Energy)")
plt.show()

In [ ]:
df['date'] = pd.to_datetime(df['date'])  # ensure datetime

df = df.set_index('date') 

df_daily = df.resample('D').mean()

plt.figure(figsize=(12,5))
plt.plot(df_daily.index, df_daily['Appliances'])
plt.title("Daily Average Energy Consumption")
plt.xlabel("Date")
plt.ylabel("Energy (Wh)")
plt.show()